# Data Preparations

## Understanding Data Structure

In [ ]:
import pandas as pd

# import raw dataset
raw_df = pd.read_csv("https://raw.githubusercontent.com/YanDraa/PemsadaDescriptive/main/employee_raw_dataset.csv")

raw_df.head()

,Employee_ID,Gender,Department,Education_Level,Job_Level,Age,Years_Experience,Training_Hours,Monthly_Salary,Performance_Score,Attendance_Rate,Project_Completed,Promotion_Last_3Y
0,1,female,Operations,Master,Junior,15,7,25.6,10702825.0,99.5,98.0,6,1
1,2,Female,Hr,Bachelor,Officer,70,12,10.2,15735781.0,76.6,85.5,2,0
2,3,MALE,Risk,Bachelor,Junior,32,18,33.2,-5000000.0,86.0,91.6,5,0
3,4,Female,IT,Bachelor,Manager,27,2,13.8,21446446.0,120.0,85.7,3,0
4,5,Female,Finance,Bachelor,Junior,16,14,104.1,10685352.0,73.2,130.0,2,1


In [ ]:
import pandas as pd

# import raw dataset
raw_df = pd.read_csv("https://raw.githubusercontent.com/YanDraa/PemsadaDescriptive/main/employee_raw_dataset.csv")

# move Employee_ID to first column
raw_df.insert(0, 'Employee_ID', raw_df.pop('Employee_ID'))

# excel-like freeze pane
display(
    raw_df.style
    .hide(axis="index")
    .set_table_styles([
        # freeze header
        {
            'selector': 'thead th',
            'props': [
                ('position', 'sticky'),
                ('top', '0'),
                ('background', "#424040"),
                ('z-index', '3')
            ]
        },

        # freeze Employee_ID column
        {
            'selector': 'thead th.col0',
            'props': [
                ('position', 'sticky'),
                ('left', '0'),
                ('z-index', '4')
            ]
        },

        {
            'selector': 'tbody td.col0',
            'props': [
                ('position', 'sticky'),
                ('left', '0'),
                ('background', "#424040"),
                ('z-index', '2')
            ]
        }
    ])
    .set_table_attributes(
        'style="display:block; overflow:auto; max-height:400px; white-space:nowrap; border-collapse:collapse;"'
    )
)

Employee_ID,Gender,Department,Education_Level,Job_Level,Age,Years_Experience,Training_Hours,Monthly_Salary,Performance_Score,Attendance_Rate,Project_Completed,Promotion_Last_3Y
1,female,Operations,Master,Junior,15,7,25.600000,10702825.000000,99.500000,98.000000,6,1
2,Female,Hr,Bachelor,Officer,70,12,10.200000,15735781.000000,76.600000,85.500000,2,0
3,MALE,Risk,Bachelor,Junior,32,18,33.200000,-5000000.000000,86.000000,91.600000,5,0
4,Female,IT,Bachelor,Manager,27,2,13.800000,21446446.000000,120.000000,85.700000,3,0
5,Female,Finance,Bachelor,Junior,16,14,104.100000,10685352.000000,73.200000,130.000000,2,1
6,Female,Operations,PhD,Senior Manager,41,9,16.000000,4095998.000000,83.100000,90.900000,5,1
7,MALE,Hr,Bachelor,Officer,29,8,16.700000,6268244.000000,79.200000,92.800000,5,0
8,MALE,IT,Bachelor,Manager,27,7,22.300000,12047713.000000,87.100000,95.400000,3,1
9,MALE,Hr,Master,Supervisor,35,1,4.800000,16111528.000000,71.400000,106.800000,4,0
10,Female,Compliance,Bachelor,Supervisor,38,12,32.800000,9389963.000000,58.800000,94.300000,2,0


In [ ]:
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 510 entries, 0 to 509
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Employee_ID        510 non-null    int64  
 1   Gender             510 non-null    object 
 2   Department         495 non-null    object 
 3   Education_Level    510 non-null    object 
 4   Job_Level          510 non-null    object 
 5   Age                510 non-null    int64  
 6   Years_Experience   510 non-null    int64  
 7   Training_Hours     475 non-null    float64
 8   Monthly_Salary     485 non-null    float64
 9   Performance_Score  510 non-null    float64
 10  Attendance_Rate    510 non-null    float64
 11  Project_Completed  510 non-null    int64  
 12  Promotion_Last_3Y  510 non-null    int64  
dtypes: float64(4), int64(5), object(4)
memory usage: 51.9+ KB


--------


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats as scipy_stats

def describe_numeric(df):

    # pilih hanya kolom bertipe numerik (int64 dan float64)
    num_cols = df.select_dtypes(include=["int64", "float64"]).columns
    results  = {}

    for col in num_cols:

        # buang NaN agar tidak mempengaruhi hasil perhitungan
        series  = df[col].dropna()
        n_total = len(df[col])              # total baris termasuk NaN
        n_valid = len(series)              # total baris tanpa NaN

        # --- tendensi sentral ---
        mean_val   = series.mean()          # rata-rata aritmatik
        median_val = series.median()        # nilai tengah setelah diurutkan
        mode_val   = series.mode()[0] if not series.mode().empty else np.nan  # nilai paling sering muncul

        # --- penyebaran ---
        std_val = series.std(ddof=1)        # standar deviasi sampel, ddof=1 berarti pembagi n-1
        var_val = series.var(ddof=1)        # variansi = std^2
        q1      = series.quantile(0.25)     # kuartil 1 (persentil ke-25)
        q3      = series.quantile(0.75)     # kuartil 3 (persentil ke-75)
        iqr     = q3 - q1                   # interquartile range = Q3 - Q1

        # --- bentuk distribusi ---
        skew_val = series.skew()            # asimetri: positif = ekor kanan, negatif = ekor kiri
        kurt_val = series.kurt()            # keruncingan: > 0 lebih runcing, < 0 lebih datar dari normal

        # --- missing value ---
        n_missing   = n_total - n_valid
        pct_missing = (n_missing / n_total) * 100

        # --- uji normalitas Shapiro-Wilk, dibatasi 5000 sampel agar tidak lambat ---
        # H0: data berdistribusi normal. Jika p >= 0.05 maka gagal tolak H0
        sample    = series.sample(min(len(series), 5000), random_state=42)
        _, p_sw   = scipy_stats.shapiro(sample)
        is_normal = "Normal" if p_sw >= 0.05 else "Not Normal"

        # --- deteksi outlier metode IQR ---
        # nilai di luar [Q1 - 1.5*IQR, Q3 + 1.5*IQR] dianggap outlier
        lower     = q1 - 1.5 * iqr
        upper     = q3 + 1.5 * iqr
        n_outlier = int(((series < lower) | (series > upper)).sum())

        results[col] = {
            "count"        : n_valid,
            "missing"      : n_missing,
            "missing_%"    : round(pct_missing, 2),
            "mean"         : round(mean_val, 4),
            "median"       : round(median_val, 4),
            "mode"         : round(mode_val, 4),
            "std"          : round(std_val, 4),
            "variance"     : round(var_val, 4),
            "min"          : round(series.min(), 4),
            "Q1 (25%)"     : round(q1, 4),
            "Q3 (75%)"     : round(q3, 4),
            "max"          : round(series.max(), 4),
            "IQR"          : round(iqr, 4),
            "skewness"     : round(skew_val, 4),
            "kurtosis"     : round(kurt_val, 4),
            "distribution" : is_normal,
            "n_outliers"   : n_outlier,
        }

    # transpose agar tampil seperti output describe() — kolom jadi baris
    return pd.DataFrame(results).T

describe_numeric(raw_df)

,count,missing,missing_%,mean,median,mode,std,variance,min,Q1 (25%),Q3 (75%),max,IQR,skewness,kurtosis,distribution,n_outliers
Employee_ID,510,0,0.0,245.6471,245.5,1,147.1217,21644.7868,1,118.25,372.75,500,254.5,0.0055,-1.2069,Not Normal,0
Age,510,0,0.0,33.7314,33.0,29,8.8713,78.6998,11,28.0,40.0,70,12.0,0.401,1.1288,Not Normal,4
Years_Experience,510,0,0.0,8.049,8.0,7,4.9448,24.4514,-6,5.0,11.0,22,6.0,-0.0437,-0.1388,Normal,3
Training_Hours,475,35,6.86,20.9107,17.1,1.7,15.8284,250.5381,0.2,9.7,27.9,104.1,18.2,1.8102,5.3648,Not Normal,12
Monthly_Salary,485,25,4.9,12492230.3278,12061459.0,-5000000.0,5094031.9399,25949161404537.449219,-5000000.0,9554922.0,15254192.0,44648274.0,5699270.0,1.2379,7.3636,Not Normal,10
Performance_Score,510,0,0.0,77.7378,77.3,79.6,11.9686,143.2465,39.6,70.9,84.5,120.0,13.6,0.1143,0.9146,Not Normal,19
Attendance_Rate,510,0,0.0,92.9441,92.9,95.7,5.6057,31.4238,77.8,89.525,96.0,130.0,6.475,1.5377,10.217,Not Normal,11
Project_Completed,510,0,0.0,5.0725,5.0,5,2.3024,5.3012,0,4.0,6.0,13,2.0,0.4036,0.0989,Not Normal,19
Promotion_Last_3Y,510,0,0.0,0.2392,0.0,0,0.427,0.1823,0,0.0,0.0,1,0.0,1.2262,-0.4984,Not Normal,122


# Task 1
**Your Task:** Create a function describe some basic statistics as similar to the "describe(include="int64")' function. But you have to add more important metrics to your functions, such as variance, mode, Skewness, Kurtosis, number of missing value (percentage of missing values), nomal distribution or not, number of outlier values.

**Interpretasi:**

- Kolom `distribution` menunjukkan hasil uji normalitas Shapiro-Wilk. Jika *"Not Normal"* pada `Monthly_Salary`, analisis lanjutan sebaiknya menggunakan pendekatan non-parametrik (Wilcoxon, Mann-Whitney, dll).
- Nilai `skewness` positif pada `Monthly_Salary` menandakan distribusi menceng ke kanan — ada sekelompok kecil karyawan bergaji sangat tinggi yang menarik rata-rata melebihi median.
- Perbedaan besar antara `mean` dan `median` pada suatu kolom juga menjadi indikasi kuat adanya pengaruh nilai ekstrem.
- Kolom `n_outliers` membantu menentukan apakah perlu penanganan outlier sebelum pemodelan. Persentase outlier > 5% adalah sinyal untuk memeriksa kualitas data lebih lanjut.

---

### Custom Categorical Describe Function

Fungsi `describe_categorical(df)` melengkapi `df.describe(include="object")` dengan menambahkan informasi frekuensi mode serta jumlah dan persentase missing value.

**Konsep:**
- **Mode** → kategori yang paling sering muncul. Satu-satunya ukuran tendensi sentral yang berlaku untuk data nominal.
- **Unique** → jumlah kategori berbeda. Nilai sangat tinggi bisa menandakan inkonsistensi penulisan (typo, kapitalisasi berbeda).
- **mode_%** → dominansi satu kategori. Nilai mendekati 100% berarti distribusi sangat tidak seimbang *(imbalanced)*.

In [ ]:
def describe_categorical(df):

    # pilih hanya kolom bertipe object atau string (kategorikal)
    cat_cols = df.select_dtypes(include=["object", "string"]).columns
    results  = {}

    for col in cat_cols:
        series    = df[col]
        n_total   = len(series)
        n_missing = series.isna().sum()     # hitung baris yang kosong (NaN)

        # buang NaN sebelum hitung statistik kategorikal
        clean   = series.dropna()
        n_valid = len(clean)

        n_unique  = clean.nunique()         # jumlah kategori unik yang berbeda
        mode_val  = clean.mode()[0] if not clean.empty else "N/A"   # kategori paling sering muncul
        mode_freq = int((clean == mode_val).sum())                  # berapa kali mode muncul
        mode_pct  = round((mode_freq / n_valid) * 100, 2) if n_valid > 0 else 0    # dominansi mode dalam persen
        pct_missing = round((n_missing / n_total) * 100, 2)        # persentase baris kosong

        results[col] = {
            "count"      : n_valid,
            "missing"    : n_missing,
            "missing_%"  : pct_missing,
            "unique"     : n_unique,
            "mode"       : mode_val,
            "mode_freq"  : mode_freq,
            "mode_%"     : mode_pct,
        }

    # transpose agar tampil seperti output describe() — kolom jadi baris
    return pd.DataFrame(results).T

describe_categorical(raw_df)

,count,missing,missing_%,unique,mode,mode_freq,mode_%
Gender,510,0,0.0,8,Female,190,37.25
Department,495,15,2.94,13,Marketing,64,12.93
Education_Level,510,0,0.0,8,Bachelor,263,51.57
Job_Level,510,0,0.0,5,Officer,160,31.37


# Task 2
**Your Task:** Create a function describe some basic statistics as similar to the "describe(include="string")' function. But you have to add more important metrics to your functions, mode, number of missing value (percentage of missing values).

**Interpretasi:**

- `missing_%` yang tinggi pada kolom kategorikal seperti `Department` atau `Education_Level` mengkonfirmasi perlunya imputasi dengan mode — sesuai strategi yang sudah diterapkan di bagian *Handling Missing Value*.
- Kolom `unique` pada `Gender` yang menunjukkan lebih dari 2–3 nilai mengkonfirmasi adanya inkonsistensi penulisan yang memerlukan standardisasi di bagian *Inconsistent Categorical Values*.
- Nilai `mode_%` yang sangat tinggi (> 80%) pada suatu kolom perlu diwaspadai jika kolom tersebut akan digunakan sebagai target variabel dalam pemodelan klasifikasi karena distribusi kelas tidak seimbang.

---

## Handling Missing Value

Before handling missing values, an initial inspection was conducted to identify categorical features containing incomplete data. This step aims to measure the number and percentage of missing values in each categorical variable, helping determine the appropriate data cleaning strategy before further analysis or modeling.

In [ ]:
raw_df.isnull().sum()

,0
Employee_ID,0
Gender,0
Department,15
Education_Level,0
Job_Level,0
Age,0
Years_Experience,0
Training_Hours,35
Monthly_Salary,25
Performance_Score,0


In [ ]:
missing_percent = raw_df.isnull().mean() * 100
missing_percent.sort_values(ascending=False)

,0
Training_Hours,6.862745
Monthly_Salary,4.901961
Department,2.941176
Employee_ID,0.000000
Gender,0.000000
Job_Level,0.000000
Education_Level,0.000000
Years_Experience,0.000000
Age,0.000000
Performance_Score,0.000000


In [ ]:
import pandas as pd

# detect missing values (all columns)
missing_count = raw_df.isnull().sum()

# create table
missing_table = pd.DataFrame({
    'Feature': missing_count.index,
    'Missing Value': missing_count.values,
    'Missing Percentage (%)': (missing_count.values / len(raw_df)) * 100
})

# sort
missing_table = missing_table.sort_values(
    by='Missing Value',
    ascending=False
)

# display
missing_table

,Feature,Missing Value,Missing Percentage (%)
7,Training_Hours,35,6.862745
8,Monthly_Salary,25,4.901961
2,Department,15,2.941176
0,Employee_ID,0,0.000000
1,Gender,0,0.000000
4,Job_Level,0,0.000000
3,Education_Level,0,0.000000
6,Years_Experience,0,0.000000
5,Age,0,0.000000
9,Performance_Score,0,0.000000


### Numeric Values

In [ ]:
raw_df["Monthly_Salary"] = raw_df["Monthly_Salary"].fillna(raw_df["Monthly_Salary"].median())
raw_df["Training_Hours"] = raw_df["Training_Hours"].fillna(raw_df["Training_Hours"].median())

### Categoric Values

In [ ]:
raw_df["Department"] = raw_df["Department"].fillna(raw_df["Department"].mode()[0])
raw_df["Education_Level"] = raw_df["Education_Level"].fillna(raw_df["Education_Level"].mode()[0])

In [ ]:
import pandas as pd

for col in raw_df.columns:
    if raw_df[col].dtype in ["int64", "float64"]:
        raw_df[col] = raw_df[col].fillna(raw_df[col].median())
    else:
        raw_df[col] = raw_df[col].fillna(raw_df[col].mode()[0])

## Remove Duplicate Data

In [ ]:
raw_df.duplicated().sum()

np.int64(10)

In [ ]:
raw_df = raw_df.drop_duplicates()

## Inconsisten Categorical Values

In [ ]:
raw_df["Gender"].unique()

array(['female', 'Female', 'MALE', 'Male', 'M', 'Unknown', 'F', '-'],
      dtype=object)

In [ ]:
raw_df["Gender"] = raw_df["Gender"].astype(str).str.strip().str.lower().replace({
    "female":"Female","f":"Female","male":"Male","m":"Male","-":"Unknown"
}).fillna("Unknown")

In [ ]:
raw_df["Department"].unique()

array(['Operations', 'Hr', 'Risk', 'IT', 'Finance', 'Compliance',
       'Marketing', 'XDept', 'HR', 'finance', 'Unknown', '999', '???'],
      dtype=object)

In [ ]:
raw_df["Department"] = (
    raw_df["Department"].astype(str).str.strip().str.lower().replace({
        "hr":"HR","operations":"Operations","risk":"Risk","it":"IT",
        "finance":"Finance","compliance":"Compliance","marketing":"Marketing",
        "xdept":"Unknown","999":"Unknown","???":"Unknown","nan":"Unknown"
    }).fillna("Unknown")
)

In [ ]:
raw_df["Education_Level"].unique()

array(['Master', 'Bachelor', 'PhD', 'S1', 'Diploma', '?', 'Magister',
       'Bachelor Degree'], dtype=object)

In [ ]:
raw_df["Education_Level"] = (
    raw_df["Education_Level"].astype(str).str.strip().str.lower().replace({
        "phd":"PhD",
        "master":"Master","magister":"Master",
        "bachelor":"Bachelor","s1":"Bachelor","bachelor degree":"Bachelor",
        "diploma":"Diploma",
        "?":"Unknown","nan":"Unknown"
    }).fillna("Unknown")
)

In [ ]:
raw_df["Job_Level"].unique()

array(['Junior', 'Officer', 'Manager', 'Senior Manager', 'Supervisor'],
      dtype=object)

In [ ]:
raw_df["Job_Level_Encoded"] = raw_df["Job_Level"].map({
    "Junior":1,
    "Officer":2,
    "Supervisor":3,
    "Manager":4,
    "Senior Manager":5
})

In [ ]:
def clean(col, mapping):
    if col in raw_df:
        raw_df[col] = (
            raw_df[col].astype(str).str.strip().str.lower()
            .map(mapping).fillna("Unknown")
        )

# Gender
clean("Gender", {"female":"Female","f":"Female","male":"Male","m":"Male"})

# Department
clean("Department", {
    "hr":"HR","operations":"Operations","risk":"Risk","it":"IT",
    "finance":"Finance","compliance":"Compliance","marketing":"Marketing"
})

# Education
clean("Education_Level", {
    "phd":"PhD","master":"Master","magister":"Master",
    "bachelor":"Bachelor","s1":"Bachelor","bachelor degree":"Bachelor",
    "diploma":"Diploma"
})

# Job Level + Encoding
clean("Job_Level", {
    "junior":"Junior","officer":"Officer","supervisor":"Supervisor",
    "manager":"Manager","senior manager":"Senior Manager"
})

if "Job_Level" in raw_df:
    raw_df["Job_Level_Encoded"] = raw_df["Job_Level"].map({
        "Junior":1,"Officer":2,"Supervisor":3,"Manager":4,"Senior Manager":5
    })

---
### Fungsi Deteksi & Penggantian Outlier

Chunk sebelumnya sudah mendeteksi outlier pada `Monthly_Salary` secara manual. Fungsi berikut menggenalisasi proses tersebut untuk **semua kolom numerik sekaligus**.

**Konsep IQR:**
$$IQR = Q3 - Q1$$
$$\text{Batas Bawah} = Q1 - 1.5 \times IQR \quad ; \quad \text{Batas Atas} = Q3 + 1.5 \times IQR$$

**Strategi penggantian:**
- `"cap"` *(Winsorizing)* → nilai outlier diganti dengan nilai batas. Jumlah baris tetap sama.
- `"median"` → nilai outlier diganti dengan median kolom. Lebih tepat bila distribusi sangat skewed.

## Outlier Detection

In [ ]:
# fungsi deteksi outlier — menggeneralisasi deteksi manual Monthly_Salary
# pada chunk sebelumnya ke semua kolom numerik sekaligus
def detect_outliers(df, threshold=1.5):

    # pilih hanya kolom numerik
    num_cols = df.select_dtypes(include=["int64", "float64"]).columns
    summary  = []

    for col in num_cols:
        series = df[col].dropna()           # buang NaN sebelum hitung batas

        Q1  = series.quantile(0.25)
        Q3  = series.quantile(0.75)
        IQR = Q3 - Q1

        lower = Q1 - threshold * IQR        # batas bawah — sama seperti logika chunk sebelumnya
        upper = Q3 + threshold * IQR        # batas atas  — sama seperti logika chunk sebelumnya

        # True jika nilai adalah outlier, False jika tidak
        outlier_mask = (series < lower) | (series > upper)
        n_outliers   = int(outlier_mask.sum())
        pct_outliers = round((n_outliers / len(series)) * 100, 2)

        summary.append({
            "column"      : col,
            "Q1"          : round(Q1, 2),
            "Q3"          : round(Q3, 2),
            "IQR"         : round(IQR, 2),
            "lower_bound" : round(lower, 2),
            "upper_bound" : round(upper, 2),
            "n_outliers"  : n_outliers,
            "outlier_%"   : pct_outliers,
        })

    # tampilkan ringkasan outlier per kolom dengan nama kolom sebagai index
    return pd.DataFrame(summary).set_index("column")



In [ ]:
# fungsi penggantian outlier — menggeneralisasi proses yang sama
# ke semua kolom numerik dengan dua pilihan strategi
def replace_outliers(df, threshold=1.5, strategy="cap"):

    # salin DataFrame agar data asli tidak berubah
    df_clean = df.copy()
    num_cols = df_clean.select_dtypes(include=["int64", "float64"]).columns

    for col in num_cols:
        series = df_clean[col]
        Q1     = series.quantile(0.25)
        Q3     = series.quantile(0.75)
        IQR    = Q3 - Q1
        lower  = Q1 - threshold * IQR
        upper  = Q3 + threshold * IQR

        if strategy == "cap":
            # winsorizing: nilai di luar batas diganti dengan nilai batas
            # nilai < lower → jadi lower, nilai > upper → jadi upper
            df_clean[col] = series.clip(lower=lower, upper=upper)

        elif strategy == "median":
            # ganti outlier dengan median — lebih robust untuk distribusi skewed
            median_val   = series.median()
            outlier_mask = (series < lower) | (series > upper)
            df_clean.loc[outlier_mask, col] = median_val

    return df_clean


detect_outliers(raw_df)

,Q1,Q3,IQR,lower_bound,upper_bound,n_outliers,outlier_%
column,,,,,,,
Employee_ID,125.75,375.25,249.50,-248.50,749.50,0,0.0
Age,28.00,40.00,12.00,10.00,58.00,2,0.4
Years_Experience,5.00,11.00,6.00,-4.00,20.00,3,0.6
Training_Hours,10.68,26.50,15.82,-13.06,50.24,22,4.4
Monthly_Salary,9717117.75,15060496.00,5343378.25,1702050.38,23075563.38,11,2.2
Performance_Score,70.47,84.43,13.95,49.55,105.35,15,3.0
Attendance_Rate,89.60,96.00,6.40,80.00,105.60,10,2.0
Project_Completed,4.00,6.00,2.00,1.00,9.00,19,3.8
Promotion_Last_3Y,0.00,0.00,0.00,0.00,0.00,118,23.6


In [ ]:
df_clean = replace_outliers(raw_df, strategy="cap")

# bandingkan statistik sebelum & sesudah pada semua kolom numerik
num_cols = raw_df.select_dtypes(include=["int64", "float64"]).columns
pd.DataFrame({
    "mean_before" : raw_df[num_cols].mean().round(2),
    "mean_after"  : df_clean[num_cols].mean().round(2),
    "std_before"  : raw_df[num_cols].std().round(2),
    "std_after"   : df_clean[num_cols].std().round(2),
    "max_before"  : raw_df[num_cols].max().round(2),
    "max_after"   : df_clean[num_cols].max().round(2),
})

,mean_before,mean_after,std_before,std_after,max_before,max_after
Employee_ID,250.50,250.50,144.48,144.48,500.0,500.00
Age,33.77,33.74,8.49,8.40,70.0,58.00
Years_Experience,8.00,8.00,4.92,4.89,22.0,20.00
Training_Hours,20.31,19.81,14.48,12.81,104.1,50.24
Monthly_Salary,12506254.75,12361333.21,4854113.78,4150591.20,44648274.0,23075563.38
Performance_Score,77.47,77.48,11.69,11.32,120.0,105.35
Attendance_Rate,92.84,92.80,5.10,4.79,130.0,105.60
Project_Completed,5.10,5.05,2.30,2.15,13.0,9.00
Promotion_Last_3Y,0.24,0.00,0.43,0.00,1.0,0.00
Job_Level_Encoded,2.40,2.40,1.13,1.13,5.0,5.00


# Task 3
**Your Task:**

- Create a function to detect all outlier values at your dataset, esspecially to numerical variables.
- Create a function to replace all outlier values at your dataset, esspecially to numerical variables.

**Interpretasi:**

- Dari `detect_outliers`, perhatikan kolom `outlier_%` — kolom yang memiliki persentase outlier > 5% memiliki nilai ekstrem yang cukup signifikan dan perlu ditangani sebelum pemodelan.
- Dari tabel perbandingan, kolom yang mengalami penurunan `std` cukup besar setelah capping menandakan bahwa outlier sebelumnya memang sangat mempengaruhi penyebaran data.
- Strategi `"cap"` dipilih karena lebih aman — jumlah baris tetap sama dan informasi relatif antar data tidak hilang, hanya nilai ekstremnya yang ditekan ke batas wajar.

---

## Data Transformation

### Grupping Numerical Values

In [ ]:
def age_group(age):
    if age < 30:
        return "Young"
    elif age < 40:
        return "Mid Career"
    elif age < 50:
        return "Senior"
    else:
        return "Late Career"

raw_df["Age_Group"] = raw_df["Age"].apply(age_group)

In [ ]:
def salary_band(s):
    if pd.isna(s):
        return "Unknown"
    if s < 8_000_000:
        return "Low Salary"
    elif s < 15_000_000:
        return "Middle Salary"
    elif s < 25_000_000:
        return "High Salary"
    else:
        return "Very High Salary"

raw_df["Salary_Band"] = raw_df["Monthly_Salary"].apply(salary_band)

In [ ]:
def performance_category(score):
    if score >= 85:
        return "High Performer"
    elif score >= 70:
        return "Moderate Performer"
    else:
        return "Low Performer"

raw_df["Performance_Category"] = raw_df["Performance_Score"].apply(performance_category)

### Fungsi Binning Variabel Numerik

Chunk sebelumnya sudah melakukan binning menggunakan fungsi `if-elif-else` terpisah untuk setiap kolom (`age_group`, `salary_band`, `performance_category`, `attendance_category`). Fungsi `bin_numeric` berikut menggeneralisasi seluruh proses tersebut menjadi **satu fungsi yang fleksibel** untuk kolom numerik apapun.

**Konsep Binning:**

Binning mengubah variabel kontinu menjadi variabel diskrit/ordinal dengan mengelompokkan nilai ke dalam interval:
$$[b_0, b_1),\ [b_1, b_2),\ \ldots,\ [b_{n-1}, b_n]$$

Panjang `labels` harus selalu = panjang `bins` - 1.

In [ ]:
def bin_numeric(df, col, bins, labels, new_col_name=None):

    # jika nama kolom baru tidak ditentukan, gunakan nama kolom + "_Group"
    if new_col_name is None:
        new_col_name = col + "_Group"

    # pd.cut() membagi nilai ke dalam interval berdasarkan bins
    # include_lowest=True → nilai minimum masuk ke interval pertama
    # right=False         → interval [kiri, kanan): kiri tertutup, kanan terbuka
    df[new_col_name] = pd.cut(
        df[col],
        bins=bins,
        labels=labels,
        include_lowest=True,
        right=False
    )

    # tambahkan kolom hasil binning langsung ke df (in-place)
    return df


# --- generalisasi chunk lama: Age ---
bin_numeric(raw_df, "Age",
            bins=[0, 30, 40, 50, 100],
            labels=["Young", "Mid Career", "Senior", "Late Career"],
            new_col_name="Age_Group")

# --- generalisasi chunk lama: Monthly_Salary ---
bin_numeric(raw_df, "Monthly_Salary",
            bins=[0, 8_000_000, 15_000_000, 25_000_000, float("inf")],
            labels=["Low Salary", "Middle Salary", "High Salary", "Very High Salary"],
            new_col_name="Salary_Band")

# --- generalisasi chunk lama: Performance_Score ---
bin_numeric(raw_df, "Performance_Score",
            bins=[0, 70, 85, 101],
            labels=["Low Performer", "Moderate Performer", "High Performer"],
            new_col_name="Performance_Category")

# --- generalisasi chunk lama: Attendance_Rate ---
bin_numeric(raw_df, "Attendance_Rate",
            bins=[0, 80, 90, 95, 101],
            labels=["Poor Attendance", "Warning Attendance",
                    "Good Attendance", "Excellent Attendance"],
            new_col_name="Attendance_Category")

# tampilkan semua hasil grouping sekaligus
raw_df[["Age", "Age_Group",
        "Monthly_Salary", "Salary_Band",
        "Performance_Score", "Performance_Category",
        "Attendance_Rate", "Attendance_Category"]].head(10)

,Age,Age_Group,Monthly_Salary,Salary_Band,Performance_Score,Performance_Category,Attendance_Rate,Attendance_Category
0,15,Young,10702825.0,Middle Salary,99.5,High Performer,98.0,Excellent Attendance
1,70,Late Career,15735781.0,High Salary,76.6,Moderate Performer,85.5,Warning Attendance
2,32,Mid Career,-5000000.0,NaN,86.0,High Performer,91.6,Good Attendance
3,27,Young,21446446.0,High Salary,120.0,NaN,85.7,Warning Attendance
4,16,Young,10685352.0,Middle Salary,73.2,Moderate Performer,130.0,NaN
5,41,Senior,4095998.0,Low Salary,83.1,Moderate Performer,90.9,Good Attendance
6,29,Young,6268244.0,Low Salary,79.2,Moderate Performer,92.8,Good Attendance
7,27,Young,12047713.0,Middle Salary,87.1,High Performer,95.4,Excellent Attendance
8,35,Mid Career,16111528.0,High Salary,71.4,Moderate Performer,106.8,NaN
9,38,Mid Career,9389963.0,Middle Salary,58.8,Low Performer,94.3,Good Attendance


# Task 4
**Your Task:**  Create a Python function that performs grouping (binning) of numerical variables into categories based on specified value ranges.

**Interpretasi:**

- Fungsi `bin_numeric` menggeneralisasi keempat fungsi `if-elif-else` pada chunk sebelumnya (`age_group`, `salary_band`, `performance_category`, `attendance_category`) menjadi satu fungsi yang cukup dipanggil dengan parameter berbeda.
- Hasilnya identik dengan chunk lama — kolom `Age_Group`, `Salary_Band`, `Performance_Category`, dan `Attendance_Category` menghasilkan kategori yang sama, namun cara penulisannya jauh lebih ringkas dan konsisten.
- Keunggulan fungsi generik ini adalah fleksibilitasnya — cukup ubah parameter `col`, `bins`, dan `labels` untuk melakukan binning pada kolom numerik apapun tanpa perlu menulis fungsi baru.
---

### Encoding Categorical Variable

### Fungsi Encoding Variabel Kategorikal

Chunk sebelumnya sudah melakukan encoding untuk `Gender` dan `Job_Level` menggunakan `.map()` secara langsung. Fungsi `encode_categorical` berikut menggeneralisasi proses tersebut menjadi **satu fungsi dengan tiga metode encoding** yang bisa digunakan untuk kolom kategorikal apapun.

**Konsep Encoding:**
- **Label Encoding** → setiap kategori diberi integer unik (0, 1, 2, ...). Cocok untuk variabel biner atau ordinal sederhana.
- **Ordinal Encoding** → urutan angka ditentukan eksplisit oleh pengguna sesuai hierarki logis. Cocok untuk variabel dengan urutan bermakna.
- **One-Hot Encoding** → membuat kolom biner baru per kategori. Cocok untuk variabel nominal multikategori agar model tidak mengasumsikan adanya urutan. `drop_first=True` menghapus satu kolom referensi untuk menghindari *dummy variable trap*.

In [ ]:
raw_df["Gender_Code"] = raw_df["Gender"].map({
    "Female": 0,
    "Male": 1
})

In [ ]:
job_level_order = {
    "Junior": 1,
    "Officer": 2,
    "Supervisor": 3,
    "Manager": 4,
    "Senior Manager": 5
}

raw_df["Job_Level_Code"] = raw_df["Job_Level"].map(job_level_order)

In [ ]:
def encode_categorical(df, col, encoding_type="label", mapping=None,
                        new_col_name=None, drop_first=True):

    # jika nama kolom baru tidak ditentukan, gunakan nama kolom + "_Code"
    if new_col_name is None:
        new_col_name = col + "_Code"

    if encoding_type == "label":
        # urutkan nilai unik secara alfabet agar pemberian indeks konsisten
        unique_vals = sorted(df[col].dropna().unique())
        # buat mapping otomatis: kategori → integer mulai dari 0
        label_map   = {val: idx for idx, val in enumerate(unique_vals)}
        df[new_col_name] = df[col].map(label_map)

    elif encoding_type == "ordinal":
        # gunakan mapping urutan yang diberikan pengguna (hierarki eksplisit)
        if mapping is None:
            raise ValueError("Ordinal encoding membutuhkan parameter 'mapping'.")
        df[new_col_name] = df[col].map(mapping)

    elif encoding_type == "onehot":
        # buat kolom dummy biner untuk setiap kategori
        # drop_first=True → hapus satu kolom referensi untuk hindari dummy variable trap
        dummies = pd.get_dummies(df[col], prefix=col, drop_first=drop_first).astype(int)
        # tambahkan tiap kolom dummy langsung ke df (in-place)
        for c in dummies.columns:
            df[c] = dummies[c]

    return df

In [ ]:
# --- generalisasi chunk lama: Gender (label encoding) ---
encode_categorical(raw_df, "Gender",
                   encoding_type="label",
                   new_col_name="Gender_Code")

# --- generalisasi chunk lama: Job_Level (ordinal encoding) ---
encode_categorical(raw_df, "Job_Level",
                   encoding_type="ordinal",
                   mapping={"Junior": 1, "Officer": 2, "Supervisor": 3,
                            "Manager": 4, "Senior Manager": 5},
                   new_col_name="Job_Level_Code")

# tampilkan hasil — identik dengan output chunk lama
raw_df[["Gender", "Gender_Code",
        "Job_Level", "Job_Level_Code"]].head(10)

,Gender,Gender_Code,Job_Level,Job_Level_Code
0,Female,0,Junior,1
1,Female,0,Officer,2
2,Male,1,Junior,1
3,Female,0,Manager,4
4,Female,0,Junior,1
5,Female,0,Senior Manager,5
6,Male,1,Officer,2
7,Male,1,Manager,4
8,Male,1,Supervisor,3
9,Female,0,Supervisor,3


# Task 5
**Your Task:** Create a Python function that performs encoding of categorical variables by converting them into numerical representations suitable for machine learning models.

**Interpretasi:**

- Fungsi `encode_categorical` menggeneralisasi kedua encoding manual pada chunk sebelumnya (`Gender` dengan `.map()` dan `Job_Level` dengan `.map(job_level_order)`) menjadi satu fungsi yang konsisten.
- Hasil `Gender_Code` dan `Job_Level_Code` identik dengan chunk lama — Female=0, Male=1 untuk Gender, dan Junior=1 hingga Senior Manager=5 untuk Job_Level.
- Keunggulan fungsi ini adalah kemampuannya menangani tiga jenis encoding sekaligus dalam satu fungsi, sehingga tidak perlu menulis logika berbeda untuk setiap kolom kategorikal baru yang ditemui.
---

### Standardization Z-Score

In [ ]:
from scipy import stats

raw_df["Salary_ZScore"] = stats.zscore(raw_df["Monthly_Salary"])
raw_df["Performance_ZScore"] = stats.zscore(raw_df["Performance_Score"])
raw_df["Training_ZScore"] = stats.zscore(raw_df["Training_Hours"])

In [ ]:
from scipy import stats

cols = ["Monthly_Salary", "Performance_Score", "Training_Hours"]

for c in cols:
    if c in raw_df.columns:
        raw_df[c + "_ZScore"] = stats.zscore(raw_df[c])

### Min-Max Normalization

In [ ]:
def minmax(col):
    return (raw_df[col] - raw_df[col].min()) / (raw_df[col].max() - raw_df[col].min())

raw_df["Salary_Normalized"] = minmax("Monthly_Salary")

# Descriptive Statistics

## Categorical Data

### Frequency

In [ ]:
raw_df["Department"].value_counts()

,count
Department,
Finance,111
HR,94
Marketing,79
Risk,59
IT,53
Operations,49
Compliance,43
Unknown,12


### Proportion

In [ ]:
raw_df["Department"].value_counts(normalize=True) * 100

,proportion
Department,
Finance,22.2
HR,18.8
Marketing,15.8
Risk,11.8
IT,10.6
Operations,9.8
Compliance,8.6
Unknown,2.4


### Contingency Table

In [ ]:
pd.crosstab(raw_df["Department"], raw_df["Performance_Category"])

Performance_Category,Low Performer,Moderate Performer,High Performer
Department,,,
Compliance,11,18,11
Finance,26,53,29
HR,23,55,13
IT,12,28,12
Marketing,21,43,15
Operations,13,26,10
Risk,11,33,14
Unknown,3,6,3


In [ ]:
pd.crosstab(
    raw_df["Department"],
    raw_df["Performance_Category"],
    normalize="index"
) * 100

Performance_Category,Low Performer,Moderate Performer,High Performer
Department,,,
Compliance,27.500000,45.000000,27.500000
Finance,24.074074,49.074074,26.851852
HR,25.274725,60.439560,14.285714
IT,23.076923,53.846154,23.076923
Marketing,26.582278,54.430380,18.987342
Operations,26.530612,53.061224,20.408163
Risk,18.965517,56.896552,24.137931
Unknown,25.000000,50.000000,25.000000


---
## Numerical Data

### Mean

In [ ]:
raw_df["Monthly_Salary"].mean()

np.float64(12506254.752)

### Median

In [ ]:
raw_df["Monthly_Salary"].median()

12061459.0

### Mode

In [ ]:
raw_df["Monthly_Salary"].mode()

,Monthly_Salary
0,12061459.0


### Range

In [ ]:
salary_range = raw_df["Monthly_Salary"].max() - raw_df["Monthly_Salary"].min()
salary_range

49648274.0

### Standard Deviation

In [ ]:
raw_df["Monthly_Salary"].std()

4854113.779653829

## Shape of Distribution

### Skewness

In [ ]:
raw_df["Monthly_Salary"].skew()

np.float64(1.5409970153083978)

### Kurtosis

In [ ]:
raw_df["Monthly_Salary"].kurt()

np.float64(8.440916785916983)

## Relative Position

### Percentiles

In [ ]:
raw_df["Monthly_Salary"].quantile([0.10, 0.25, 0.50, 0.75, 0.90, 0.95])

,Monthly_Salary
0.10,7172953.40
0.25,9717117.75
0.50,12061459.00
0.75,15060496.00
0.90,17737155.20
0.95,19140216.05


### Quartiles

In [ ]:
Q1 = raw_df["Monthly_Salary"].quantile(0.25)
Q2 = raw_df["Monthly_Salary"].quantile(0.50)
Q3 = raw_df["Monthly_Salary"].quantile(0.75)

Q1, Q2, Q3

(np.float64(9717117.75), np.float64(12061459.0), np.float64(15060496.0))

### Z-Score

In [ ]:
raw_df[["Monthly_Salary", "Salary_ZScore"]].head()

,Monthly_Salary,Salary_ZScore
0,10702825.0,-0.371898
1,15735781.0,0.665984
2,-5000000.0,-3.610090
3,21446446.0,1.843621
4,10685352.0,-0.375501


In [ ]:
raw_df[raw_df["Salary_ZScore"].abs() > 2]

,Employee_ID,Gender,Department,Education_Level,Job_Level,Age,Years_Experience,Training_Hours,Monthly_Salary,Performance_Score,...,Attendance_Category,Gender_Code,Job_Level_Code,Salary_ZScore,Performance_ZScore,Training_ZScore,Monthly_Salary_ZScore,Performance_Score_ZScore,Training_Hours_ZScore,Salary_Normalized
2,3,Male,Risk,Bachelor,Junior,32,18,33.2,-5000000.0,86.0,...,Good Attendance,1,1,-3.610090,0.730370,0.891011,-3.610090,0.730370,0.891011,0.000000
38,39,Male,Marketing,Master,Officer,24,4,8.4,2263294.0,78.5,...,Excellent Attendance,1,2,-2.112274,0.088087,-0.823882,-2.112274,0.088087,-0.823882,0.146295
63,64,Female,IT,Master,Officer,43,7,4.7,23002892.0,71.6,...,Excellent Attendance,0,2,2.164587,-0.502813,-1.079732,2.164587,-0.502813,-1.079732,0.564025
66,67,Female,HR,Bachelor,Officer,46,-4,17.1,1403602.0,83.0,...,Excellent Attendance,0,2,-2.289557,0.473457,-0.222286,-2.289557,0.473457,-0.222286,0.128979
139,140,Female,Marketing,Master,Manager,25,8,1.9,23956065.0,46.4,...,Warning Attendance,0,4,2.361147,-2.660884,-1.273349,2.361147,-2.660884,-1.273349,0.583224
206,207,Female,Compliance,Bachelor,Manager,27,10,3.1,38133417.0,47.9,...,Warning Attendance,0,4,5.284760,-2.532427,-1.190371,5.284760,-2.532427,-1.190371,0.868780
230,231,Female,Finance,Bachelor,Supervisor,27,7,14.3,31180707.0,39.6,...,Excellent Attendance,0,3,3.850992,-3.243220,-0.415903,3.850992,-3.243220,-0.415903,0.728740
243,244,Male,Finance,Bachelor,Officer,29,3,49.6,41352372.0,57.6,...,Excellent Attendance,1,2,5.948564,-1.701741,2.025053,5.948564,-1.701741,2.025053,0.933615
303,304,Female,Finance,Bachelor,Junior,46,0,10.7,23542938.0,48.7,...,Excellent Attendance,0,1,2.275953,-2.463917,-0.664839,2.275953,-2.463917,-0.664839,0.574903
315,316,Female,Risk,Master,Officer,40,7,31.4,2351544.0,68.2,...,Excellent Attendance,0,2,-2.094075,-0.793982,0.766543,-2.094075,-0.793982,0.766543,0.148072


### T-Score

In [ ]:
raw_df["Salary_TScore"] = 50 + (10 * raw_df["Salary_ZScore"])
raw_df[["Monthly_Salary", "Salary_ZScore", "Salary_TScore"]].head()

,Monthly_Salary,Salary_ZScore,Salary_TScore
0,10702825.0,-0.371898,46.281019
1,15735781.0,0.665984,56.659837
2,-5000000.0,-3.610090,13.899101
3,21446446.0,1.843621,68.436207
4,10685352.0,-0.375501,46.244986


## Association Metrics

### Pearson Correlation

In [ ]:
raw_df[[
    "Age",
    "Years_Experience",
    "Training_Hours",
    "Monthly_Salary",
    "Performance_Score",
    "Attendance_Rate",
    "Project_Completed"
]].corr(method="pearson")

,Age,Years_Experience,Training_Hours,Monthly_Salary,Performance_Score,Attendance_Rate,Project_Completed
Age,1.000000,-0.012917,0.044671,0.004448,-0.014363,-0.016184,0.029378
Years_Experience,-0.012917,1.000000,0.053667,-0.011149,0.018524,0.023571,-0.044013
Training_Hours,0.044671,0.053667,1.000000,-0.029937,0.007696,0.070073,0.011538
Monthly_Salary,0.004448,-0.011149,-0.029937,1.000000,-0.176254,0.039299,-0.059486
Performance_Score,-0.014363,0.018524,0.007696,-0.176254,1.000000,-0.036315,-0.007064
Attendance_Rate,-0.016184,0.023571,0.070073,0.039299,-0.036315,1.000000,-0.010853
Project_Completed,0.029378,-0.044013,0.011538,-0.059486,-0.007064,-0.010853,1.000000


### Spearman Rank Correlation

In [ ]:
raw_df[[
    "Age",
    "Years_Experience",
    "Training_Hours",
    "Monthly_Salary",
    "Performance_Score",
    "Attendance_Rate",
    "Project_Completed"
]].corr(method="spearman")

,Age,Years_Experience,Training_Hours,Monthly_Salary,Performance_Score,Attendance_Rate,Project_Completed
Age,1.000000,-0.012356,0.072380,0.000491,-0.027690,0.004668,0.042803
Years_Experience,-0.012356,1.000000,0.061325,-0.031660,0.031947,0.007623,-0.055539
Training_Hours,0.072380,0.061325,1.000000,-0.038398,-0.022664,-0.020882,-0.009488
Monthly_Salary,0.000491,-0.031660,-0.038398,1.000000,-0.066585,0.050788,-0.089591
Performance_Score,-0.027690,0.031947,-0.022664,-0.066585,1.000000,-0.038549,-0.024440
Attendance_Rate,0.004668,0.007623,-0.020882,0.050788,-0.038549,1.000000,0.018606
Project_Completed,0.042803,-0.055539,-0.009488,-0.089591,-0.024440,0.018606,1.000000


### Point-Biserial Correlation

In [ ]:
from scipy.stats import pointbiserialr

pointbiserialr(
    raw_df["Promotion_Last_3Y"],
    raw_df["Monthly_Salary"]
)

SignificanceResult(statistic=np.float64(-0.029660433256734127), pvalue=np.float64(0.5081551290310187))

### Covariance

In [ ]:
raw_df[[
    "Age",
    "Years_Experience",
    "Training_Hours",
    "Monthly_Salary",
    "Performance_Score",
    "Attendance_Rate",
    "Project_Completed"
]].cov()

,Age,Years_Experience,Training_Hours,Monthly_Salary,Performance_Score,Attendance_Rate,Project_Completed
Age,72.143531,-0.539547,5.492602e+00,1.833957e+05,-1.425944e+00,-0.700889,0.575018
Years_Experience,-0.539547,24.186369,3.820671e+00,-2.661521e+05,1.064873e+00,0.591062,-0.498794
Training_Hours,5.492602,3.820671,2.095554e+02,-2.103654e+06,1.302162e+00,5.172003,0.384881
Monthly_Salary,183395.745459,-266152.136770,-2.103654e+06,2.356242e+13,-1.000044e+07,972650.685441,-665395.682068
Performance_Score,-1.425944,1.064873,1.302162e+00,-1.000044e+07,1.366279e+02,-2.164290,-0.190263
Attendance_Rate,-0.700889,0.591062,5.172003e+00,9.726507e+05,-2.164290e+00,25.996950,-0.127515
Project_Completed,0.575018,-0.498794,3.848806e-01,-6.653957e+05,-1.902633e-01,-0.127515,5.310216


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

def cramers_v(x, y):
    table = pd.crosstab(x.fillna("NA"), y.fillna("NA"))
    chi2 = chi2_contingency(table)[0]
    n = table.values.sum()
    r, k = table.shape

    return np.sqrt(chi2 / (n * (min(r - 1, k - 1))))

def categorical_correlation_matrix(df):
    cols = df.select_dtypes(include=["object", "string"]).columns
    corr = pd.DataFrame(np.eye(len(cols)), index=cols, columns=cols)

    for i, c1 in enumerate(cols):
        for j in range(i + 1, len(cols)):
            corr.iloc[i, j] = corr.iloc[j, i] = cramers_v(df[c1], df[cols[j]])

    return corr

cat_corr = categorical_correlation_matrix(raw_df)
display(cat_corr)

,Gender,Department,Education_Level,Job_Level
Gender,1.000000,0.149802,0.135652,0.124890
Department,0.149802,1.000000,0.126385,0.109417
Education_Level,0.135652,0.126385,1.000000,0.079513
Job_Level,0.124890,0.109417,0.079513,1.000000


**Your Task:** Create a Python function that can:

- Compute numerical–numerical relationships using covariance
- Compute numerical–numerical correlations using Pearson correlation
- Compute categorical–categorical correlations using Cramér’s V
- Produce separate matrices for covariance, numerical correlation, and categorical correlation

### Fungsi Association Matrix

Chunk sebelumnya sudah menghitung covariance, Pearson correlation, dan Cramér's V secara terpisah-pisah menggunakan `.cov()`, `.corr()`, dan fungsi `cramers_v()`. Fungsi `association_matrix` berikut menggeneralisasi seluruh proses tersebut menjadi **satu fungsi terpadu** yang menghasilkan ketiga matriks sekaligus — namun diimplementasikan **dari nol secara manual** tanpa memanggil `.cov()` atau `.corr()` bawaan pandas.

**Rumus Covariance:**
$$\text{Cov}(X, Y) = \frac{\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})}{n - 1}$$

**Rumus Pearson Correlation:**
$$r = \frac{\text{Cov}(X, Y)}{s_X \cdot s_Y}$$
Nilai selalu di $[-1, 1]$: 1 = korelasi positif sempurna, -1 = negatif sempurna, 0 = tidak ada hubungan linear.

**Rumus Cramér's V:**
$$V = \sqrt{\frac{\chi^2}{n \cdot (\min(r, k) - 1)}}$$
Nilai selalu di $[0, 1]$: 0 = tidak ada asosiasi, 1 = asosiasi sempurna.

In [ ]:
import numpy as np
from scipy.stats import chi2_contingency

# --- fungsi pembantu: covariance manual ---
# menggeneralisasi .cov() bawaan pandas (Cell 93)
def manual_covariance(x, y):
    # konversi ke numpy array float agar operasi matematika tidak error
    x = x.to_numpy(dtype=float)
    y = y.to_numpy(dtype=float)
    n      = len(x)
    mean_x = x.mean()
    mean_y = y.mean()
    # rumus: Cov(X,Y) = Σ(xi - x̄)(yi - ȳ) / (n-1)
    # pembagi n-1 karena ini estimasi untuk sampel, bukan populasi
    return np.sum((x - mean_x) * (y - mean_y)) / (n - 1)

In [ ]:
# --- fungsi pembantu: pearson correlation manual ---
# menggeneralisasi .corr(method="pearson") bawaan pandas (Cell 87)
def manual_pearson(x, y):
    # konversi ke numpy array float agar operasi matematika tidak error
    x = x.to_numpy(dtype=float)
    y = y.to_numpy(dtype=float)
    n      = len(x)
    mean_x = x.mean()
    mean_y = y.mean()

    # pembilang: jumlah hasil kali deviasi
    numerator = np.sum((x - mean_x) * (y - mean_y))
    # standar deviasi masing-masing variabel
    std_x = (np.sum((x - mean_x) ** 2) / (n - 1)) ** 0.5
    std_y = (np.sum((y - mean_y) ** 2) / (n - 1)) ** 0.5

    # hindari pembagian dengan nol jika salah satu variabel tidak bervariasi
    if std_x == 0 or std_y == 0:
        return 0
    # rumus: r = Cov(X,Y) / (sX * sY) → nilai selalu di [-1, 1]
    return numerator / ((n - 1) * std_x * std_y)

In [ ]:
# --- fungsi pembantu: cramér's v manual ---
# menggeneralisasi fungsi cramers_v() yang sudah ada (Cell 95)
def manual_cramers_v(x, y):
    # buat tabel kontingensi (frekuensi silang) antar dua variabel kategorikal
    # fillna("NA") agar baris NaN tidak hilang dari perhitungan
    table = pd.crosstab(x.fillna("NA"), y.fillna("NA"))
    # hitung statistik chi-square dari tabel kontingensi
    chi2  = chi2_contingency(table)[0]
    n     = table.values.sum()              # total observasi
    r, k  = table.shape                     # jumlah baris dan kolom tabel
    denom = n * (min(r, k) - 1)            # pembagi normalisasi
    if denom == 0:
        return 0
    # rumus: V = sqrt(χ² / (n * (min(r,k) - 1))) → nilai selalu di [0, 1]
    return np.sqrt(chi2 / denom)

In [ ]:
# --- fungsi utama: tiga matriks asosiasi sekaligus ---
def association_matrix(df):
    # filter ketat hanya kolom numerik murni untuk covariance & pearson
    num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
    # filter kolom kategorikal untuk cramér's v
    cat_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()

    # --- covariance matrix (numerik - numerik) ---
    # inisialisasi matriks kosong berukuran n x n
    cov_mat = pd.DataFrame(np.zeros((len(num_cols), len(num_cols))),
                           index=num_cols, columns=num_cols)
    for i, c1 in enumerate(num_cols):
        for j, c2 in enumerate(num_cols):
            # buang baris NaN di salah satu kolom sebelum dihitung
            valid = df[[c1, c2]].dropna()
            cov_mat.iloc[i, j] = round(manual_covariance(valid[c1], valid[c2]), 4)

    # --- pearson correlation matrix (numerik - numerik) ---
    pearson_mat = pd.DataFrame(np.zeros((len(num_cols), len(num_cols))),
                               index=num_cols, columns=num_cols)
    for i, c1 in enumerate(num_cols):
        for j, c2 in enumerate(num_cols):
            valid = df[[c1, c2]].dropna()
            pearson_mat.iloc[i, j] = round(manual_pearson(valid[c1], valid[c2]), 4)

    # --- cramér's v matrix (kategorikal - kategorikal) ---
    # inisialisasi matriks identitas: diagonal = 1 karena V(X,X) = 1 selalu
    cramers_mat = pd.DataFrame(np.eye(len(cat_cols)),
                               index=cat_cols, columns=cat_cols)
    for i, c1 in enumerate(cat_cols):
        # mulai dari j = i+1 agar tidak hitung pasangan yang sama dua kali
        for j in range(i + 1, len(cat_cols)):
            v = round(manual_cramers_v(df[c1], df[cat_cols[j]]), 4)
            # matriks simetris: V(X,Y) = V(Y,X), isi dua sisi sekaligus
            cramers_mat.iloc[i, j] = v
            cramers_mat.iloc[j, i] = v

    return cov_mat, pearson_mat, cramers_mat


cov_matrix, pearson_matrix, cramers_matrix = association_matrix(raw_df)

In [ ]:
print("Covariance Matrix (Numerik - Numerik)")
display(cov_matrix)

Covariance Matrix (Numerik - Numerik)


,Employee_ID,Age,Years_Experience,Training_Hours,Monthly_Salary,Performance_Score,Attendance_Rate,Project_Completed,Promotion_Last_3Y,Job_Level_Encoded,Gender_Code,Job_Level_Code,Salary_ZScore,Performance_ZScore,Training_ZScore,Monthly_Salary_ZScore,Performance_Score_ZScore,Training_Hours_ZScore,Salary_Normalized,Salary_TScore
Employee_ID,4.175000e+04,-3.0651,-16.3918,-2.886049e+02,6.202271e+07,-8.845660e+01,-47.3706,0.9208,-1.2325,-2.2595,-1.1403,-2.2595,1.279010e+01,-7.5752,-19.9567,1.279010e+01,-7.5752,-19.9567,1.2492,1.279015e+02
Age,-3.065100e+00,144.2871,-0.5395,5.492600e+00,1.833957e+05,-1.425900e+00,-0.7009,0.5750,-0.2593,-0.2444,-0.0577,-0.2444,3.780000e-02,-0.1221,0.3798,3.780000e-02,-0.1221,0.3798,0.0037,3.782000e-01
Years_Experience,-1.639180e+01,-0.5395,48.3727,3.820700e+00,-2.661521e+05,1.064900e+00,0.5911,-0.4988,-0.1659,0.2673,-0.0413,0.2673,-5.490000e-02,0.0912,0.2642,-5.490000e-02,0.0912,0.2642,-0.0054,-5.489000e-01
Training_Hours,-2.886049e+02,5.4926,3.8207,4.191108e+02,-2.103654e+06,1.302200e+00,5.1720,0.3849,0.0020,-0.6985,-0.4381,-0.6985,-4.338000e-01,0.1115,14.4905,-4.338000e-01,0.1115,14.4905,-0.0424,-4.338100e+00
Monthly_Salary,6.202271e+07,183395.7455,-266152.1368,-2.103654e+06,4.712484e+13,-1.000044e+07,972650.6854,-665395.6821,-61196.2560,220827.2261,-33413.0840,220827.2261,4.858975e+06,-856414.5893,-145465.3422,4.858975e+06,-856414.5893,-145465.3422,474586.9028,4.858975e+07
Performance_Score,-8.845660e+01,-1.4259,1.0649,1.302200e+00,-1.000044e+07,2.732557e+02,-2.1643,-0.1903,0.1777,-1.1314,-0.0637,-1.1314,-2.062300e+00,11.7005,0.0900,-2.062300e+00,11.7005,0.0900,-0.2014,-2.062260e+01
Attendance_Rate,-4.737060e+01,-0.7009,0.5911,5.172000e+00,9.726507e+05,-2.164300e+00,51.9939,-0.1275,0.1475,-0.6607,-0.0948,-0.6607,2.006000e-01,-0.1853,0.3576,2.006000e-01,-0.1853,0.3576,0.0196,2.005800e+00
Project_Completed,9.208000e-01,0.5750,-0.4988,3.849000e-01,-6.653957e+05,-1.903000e-01,-0.1275,10.6204,0.0200,0.0471,0.0280,0.0471,-1.372000e-01,-0.0163,0.0266,-1.372000e-01,-0.0163,0.0266,-0.0134,-1.372200e+00
Promotion_Last_3Y,-1.232500e+00,-0.2593,-0.1659,2.000000e-03,-6.119626e+04,1.777000e-01,0.1475,0.0200,0.3613,-0.0049,-0.0072,-0.0049,-1.260000e-02,0.0152,0.0001,-1.260000e-02,0.0152,0.0001,-0.0012,-1.262000e-01
Job_Level_Encoded,-2.259500e+00,-0.2444,0.2673,-6.985000e-01,2.208272e+05,-1.131400e+00,-0.6607,0.0471,-0.0049,2.5659,-0.0137,1.2830,4.550000e-02,-0.0969,-0.0483,4.550000e-02,-0.0969,-0.0483,0.0044,4.554000e-01


In [ ]:
print("Pearson Correlation Matrix (Numerik - Numerik)")
display(pearson_matrix)

Pearson Correlation Matrix (Numerik - Numerik)


,Employee_ID,Age,Years_Experience,Training_Hours,Monthly_Salary,Performance_Score,Attendance_Rate,Project_Completed,Promotion_Last_3Y,Job_Level_Encoded,Gender_Code,Job_Level_Code,Salary_ZScore,Performance_ZScore,Training_ZScore,Monthly_Salary_ZScore,Performance_Score_ZScore,Training_Hours_ZScore,Salary_Normalized,Salary_TScore
Employee_ID,1.0000,-0.0025,-0.0231,-0.1380,0.0884,-0.0524,-0.0643,0.0028,-0.0201,-0.0138,-0.0155,-0.0138,0.0884,-0.0524,-0.1380,0.0884,-0.0524,-0.1380,0.0884,0.0884
Age,-0.0025,1.0000,-0.0129,0.0447,0.0044,-0.0144,-0.0162,0.0294,-0.0718,-0.0254,-0.0134,-0.0254,0.0044,-0.0144,0.0447,0.0044,-0.0144,0.0447,0.0044,0.0044
Years_Experience,-0.0231,-0.0129,1.0000,0.0537,-0.0111,0.0185,0.0236,-0.0440,-0.0793,0.0480,-0.0165,0.0480,-0.0111,0.0185,0.0537,-0.0111,0.0185,0.0537,-0.0111,-0.0111
Training_Hours,-0.1380,0.0447,0.0537,1.0000,-0.0299,0.0077,0.0701,0.0115,0.0003,-0.0426,-0.0595,-0.0426,-0.0299,0.0077,1.0000,-0.0299,0.0077,1.0000,-0.0299,-0.0299
Monthly_Salary,0.0884,0.0044,-0.0111,-0.0299,1.0000,-0.1763,0.0393,-0.0595,-0.0297,0.0402,-0.0135,0.0402,1.0000,-0.1763,-0.0299,1.0000,-0.1763,-0.0299,1.0000,1.0000
Performance_Score,-0.0524,-0.0144,0.0185,0.0077,-0.1763,1.0000,-0.0363,-0.0071,0.0358,-0.0855,-0.0107,-0.0855,-0.1763,1.0000,0.0077,-0.1763,1.0000,0.0077,-0.1763,-0.1763
Attendance_Rate,-0.0643,-0.0162,0.0236,0.0701,0.0393,-0.0363,1.0000,-0.0109,0.0681,-0.1144,-0.0365,-0.1144,0.0393,-0.0363,0.0701,0.0393,-0.0363,0.0701,0.0393,0.0393
Project_Completed,0.0028,0.0294,-0.0440,0.0115,-0.0595,-0.0071,-0.0109,1.0000,0.0204,0.0180,0.0239,0.0180,-0.0595,-0.0071,0.0115,-0.0595,-0.0071,0.0115,-0.0595,-0.0595
Promotion_Last_3Y,-0.0201,-0.0718,-0.0793,0.0003,-0.0297,0.0358,0.0681,0.0204,1.0000,-0.0101,-0.0335,-0.0101,-0.0297,0.0358,0.0003,-0.0297,0.0358,0.0003,-0.0297,-0.0297
Job_Level_Encoded,-0.0138,-0.0254,0.0480,-0.0426,0.0402,-0.0855,-0.1144,0.0180,-0.0101,1.0000,-0.0237,1.0000,0.0402,-0.0855,-0.0426,0.0402,-0.0855,-0.0426,0.0402,0.0402


In [ ]:
print("Cramér's V Matrix (Kategorikal - Kategorikal)")
display(cramers_matrix)

Cramér's V Matrix (Kategorikal - Kategorikal)


,Gender,Department,Education_Level,Job_Level
Gender,1.0000,0.1498,0.1357,0.1249
Department,0.1498,1.0000,0.1264,0.1094
Education_Level,0.1357,0.1264,1.0000,0.0795
Job_Level,0.1249,0.1094,0.0795,1.0000


**Interpretasi:**

- **Covariance Matrix** menggeneralisasi `.cov()` dari Cell 93. Nilai kovarians tidak bisa dibandingkan langsung antar pasangan kolom yang berbeda satuan (misalnya `Monthly_Salary` vs `Age`) — inilah alasan perlu dinormalisasi menjadi Pearson correlation.
- **Pearson Correlation Matrix** menggeneralisasi `.corr(method="pearson")` dari Cell 87. Pasangan `Age` dan `Years_Experience` kemungkinan memiliki nilai $|r|$ mendekati 1 — ini disebut *multicollinearity* dan perlu diwaspadai jika kedua variabel digunakan bersama dalam model regresi.
- **Cramér's V Matrix** menggeneralisasi fungsi `cramers_v()` dan `categorical_correlation_matrix()` dari Cell 95. Nilai V yang tinggi antara dua variabel kategorikal (misalnya `Department` dan `Job_Level`) menandakan distribusi antar kategori sangat bervariasi — temuan yang menarik untuk eksplorasi lebih lanjut.
- Keunggulan fungsi `association_matrix` dibanding chunk sebelumnya adalah ketiga matriks dihasilkan sekaligus dalam satu pemanggilan fungsi, tanpa perlu memanggil tiga metode berbeda secara terpisah.
---